In [1]:
import pandas as pd
import numpy as np
import re
import lightgbm as lgb

In [2]:
DATA_FOLDER = "./"

df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_big_best_customers_25c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()

In [3]:
target = df.groupby(["customer_id", "product_id"])["tn"].shift(-2)
# obtengo las 20 columnas con mayor correlación con el target
correlation = df[numeric_cols].corrwith(target).abs().sort_values(ascending=False)
top_20_cols = correlation.head(20).index.tolist()
print("Top 20 columns with highest correlation to target:")
print(top_20_cols)

/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Top 20 columns with highest correlation to target:
['tn_wavelet_0_mean_lag_11', 'tn_wavelet_0_mean_lag_8', 'tn_wavelet_0_mean_lag_15', 'tn_wavelet_0_mean', 'tn_wavelet_0_mean_lag_2', 'tn_wavelet_0_mean_lag_1', 'tn_wavelet_0_mean_lag_3', 'tn_rolling_mean_12', 'tn_wavelet_0_mean_lag_6', 'tn_rolling_mean_12_lag_1', 'tn_wavelet_0_mean_lag_20', 'tn_rolling_mean_12_lag_2', 'tn_rolling_mean_12_lag_3', 'tn_rolling_mean_24', 'tn_rolling_mean_6_lag_6', 'tn_rolling_mean_24_lag_1', 'tn_wavelet_0_max_lag_11', 'tn_wavelet_0_max_lag_15', 'tn_wavelet_0_max', 'tn_wavelet_0_max_lag_2']


In [4]:
# creo el target
final_test_df = df[df["date_id"] == df["date_id"].max()]

df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)
# elimino las rows donde target es nan
df = df[df["target"].notna()]

/tmp/ipykernel_301963/3069537650.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)


In [5]:
# reemplazar todos los valores inf y -inf por np.nan
df.replace([np.inf, -np.inf], np.nan, inplace=True) 

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import BaseCrossValidator

class CustomTimeSeriesSplitter(BaseCrossValidator):
    def __init__(self, subsample_prop=0.5, test_months=1, random_state=None, random=True, quantiles=10):
        assert 0 < subsample_prop <= 1, "subsample_prop must be in (0, 1]"
        self.subsample_prop = subsample_prop
        self.test_months = test_months
        self.random_state = np.random.RandomState(random_state)
        self._sampled_series = None
        self.random = random
        self.quantiles = quantiles

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.test_months

    def split(self, X, y=None, groups=None):
        df = X.reset_index(drop=True)
        max_date = df['date_id'].max()
        pair_col = ['product_id', 'customer_id']

        # Calcular tn total y asignar deciles
        total_tn = (
            df.groupby(pair_col)['tn'].sum()
            .reset_index(name='total_tn')
            .sort_values('total_tn', ascending=False)
            .reset_index(drop=True)
        )
        total_tn['quantile'] = pd.qcut(total_tn.index, self.quantiles, labels=False)

        # Samplear series por quantil
        if not self._sampled_series:
            sampled_series = []
            for q in range(10):
                group = total_tn[total_tn['quantile'] == q]
                n = max(1, int(len(group) * self.subsample_prop))
                if self.random:
                    # Si es aleatorio, usar random_state
                    sampled = group.sample(n=n, random_state=self.random_state)
                else:
                    # los primeros N
                    sampled = group.head(n)
                sampled_series.append(sampled)
            self._sampled_series = sampled_series
        else:
            sampled_series = self._sampled_series

        sampled_series_df = pd.concat(sampled_series, ignore_index=True)

        # Marcar las series seleccionadas (más eficiente que set + apply)
        sampled_df = df.merge(sampled_series_df[pair_col], on=pair_col, how='inner')

        for i in range(self.test_months):
            test_date = max_date - i
            test_mask = df['date_id'] == test_date
            test_idx = df.index[test_mask].tolist()

            # Entrenamiento: solo series seleccionadas y fechas anteriores
            train_mask = (sampled_df['date_id'] < test_date)
            train_idx = sampled_df.index[train_mask].tolist()

            yield train_idx, test_idx

class SimpleLastDateSplitter(BaseCrossValidator):
    """Split: test = date_id máximo, train = resto. Sin copias innecesarias."""
    def get_n_splits(self, X=None, y=None, groups=None):
        return 1

    def split(self, X, y=None, groups=None):
        # No copies, solo uso la referencia
        max_date = X['date_id'].max()
        test_mask = X['date_id'] == max_date
        
        # Obtener posiciones enteras (para .iloc) en lugar de índices del DataFrame
        test_idx = np.where(test_mask)[0]
        train_idx = np.where(~test_mask)[0]
        
        yield train_idx, test_idx

In [7]:
# Reemplazo de inf por nan
df.replace([np.inf, -np.inf], np.nan, inplace=True)



# Transformar object a category (menos claves)
columns = df.select_dtypes(include=["object"]).columns.tolist()
for col in columns:
    if col not in ["product_id", "customer_id", "date_id"]:
        df[col] = df[col].astype("category")

# Eliminar columnas datetime innecesarias
datetime_cols = df.select_dtypes(include=["datetime"]).columns.tolist()
for col in datetime_cols:
    if col != "date_id":
        df.drop(columns=[col], inplace=True)

# Crear splitter con 20% y 2 meses

# Columnas a dropear
drop_cols = ["fecha", "target", "date_id"]

In [8]:

class LGBCustomMetric:
    def __init__(self, test_df, product_ids):
        # Guardamos referencias completas sin filtrar
        self.test_df = test_df[["product_id", "target"]].copy()
        self.product_ids = set(product_ids)

    def __call__(self, preds, train_data):
        # Recrear la evaluación exactamente como en la función objective
        test_df_eval = self.test_df.copy()
        test_df_eval["predictions"] = preds
        test_df_eval = test_df_eval.groupby("product_id").agg({
            "predictions": "sum",
            "target": "sum"
        }).reset_index()
        test_df_eval = test_df_eval[test_df_eval["product_id"].isin(self.product_ids)]
        total_error = np.sum(np.abs(test_df_eval["predictions"] - test_df_eval["target"])) / test_df_eval["target"].sum()
        if total_error > 10:
            print(f"Warning: High total error {total_error:.4f} detected.")
        return "total_error", total_error, False

In [9]:
df.tail()

,product_id,customer_id,fecha,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,...,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_lag_15,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max,prod_tn_rolling_mean_24_lag_1_x_tn_wavelet_0_max_lag_2,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max_lag_15,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max,prod_tn_wavelet_0_max_lag_11_x_tn_wavelet_0_max_lag_2,prod_tn_wavelet_0_max_lag_15_x_tn_wavelet_0_max,prod_tn_wavelet_0_max_lag_15_x_tn_wavelet_0_max_lag_2,prod_tn_wavelet_0_max_x_tn_wavelet_0_max_lag_2,target
819323,21290,10021,2017-06,0.0,0,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
819326,21290,10022,2017-06,0.0,0,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
819329,21290,10023,2017-06,0.0,0,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
819332,21290,10024,2017-06,0.0,0,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
819335,21290,10025,2017-06,0.0,0,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [11]:
import optuna
import numpy as np
import lightgbm as lgb

splitter = SimpleLastDateSplitter()
for fold, (train_idx, test_idx) in enumerate(splitter.split(df)):
    # HACER COPIAS para evitar modificar los datos originales
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

train_df = train_df.groupby(["product_id", "customer_id"]).filter(lambda x: len(x) >= 12)


print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}, Fold: {fold + 1}")

train = lgb.Dataset(
    train_df.drop(columns=[col for col in drop_cols if col in train_df.columns]),
    label=train_df["target"],
    categorical_feature="auto",
    #weight=(train_df["tn"] + 0.1),
)

dtest = lgb.Dataset(
    test_df.drop(columns=[col for col in drop_cols if col in test_df.columns]),
    label=test_df["target"],
    categorical_feature="auto",
)

custom_metric = LGBCustomMetric(
    test_df=test_df,
    product_ids=product_ids,
)

def objective(trial):
    total_errors = []
    num_iterations_list = []

    # Hiperparámetros a optimizar
    params = {
        'objective': 'tweedie',
        'boosting_type': 'gbdt',
        'verbose': -1,
        'metric': 'None',
        'first_metric_only': True,
        "max_depth": -1,
        "device": "cuda",
        "feature_pre_filter": False,
        "tweedie_variance_power": trial.suggest_float("tweedie_variance_power", 1.1, 1.9),
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.1),
        'num_leaves': trial.suggest_int("num_leaves", 15, 256),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
        'feature_fraction': trial.suggest_float("feature_fraction", 0.1, 1.0),
        'bagging_fraction': trial.suggest_float("bagging_fraction", 0.5, 1.0),
        'bagging_freq': trial.suggest_int("bagging_freq", 1, 10),
    }




    model = lgb.train(
        params=params,
        train_set=train,
        #num_boost_round=9999,
        num_boost_round=100, 
        feval=custom_metric,
        valid_sets=[dtest],
        callbacks=[
            lgb.early_stopping(int(400 + 4 / params["learning_rate"]), first_metric_only=True),
            lgb.log_evaluation(period=50)
        ],
    )

    # Usar una copia separada para las predicciones finales
    test_df_eval = test_df.copy()
    predictions = model.predict(
        test_df_eval.drop(columns=[col for col in drop_cols if col in test_df_eval.columns]),
        num_iteration=model.best_iteration,
    )
    test_df_eval["predictions"] = predictions


    test_df_grouped = test_df_eval.groupby("product_id").agg({
        "predictions": "sum",
        "target": "sum"
    }).reset_index()
    test_df_grouped = test_df_grouped[test_df_grouped["product_id"].isin(product_ids)]
    abs_error = np.abs(test_df_grouped["predictions"] - test_df_grouped["target"])
    total_error = np.sum(abs_error) / np.sum(test_df_grouped["target"])

    total_errors.append(total_error)
    num_iterations_list.append(model.best_iteration)

    avg_error = np.mean(total_errors)
    trial.set_user_attr("avg_num_iterations", np.mean(num_iterations_list))
    return avg_error

# Ejecutar la optimización con Optuna
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=24),
    study_name="exp_pipeline_lgb_60_trial_no_scaling_3",
    storage="sqlite:///optuna_study.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=60)

Train shape: (698152, 681), Test shape: (23998, 681), Fold: 1


[I 2025-07-13 12:36:15,444] Using an existing study with name 'exp_pipeline_lgb_60_trial_no_scaling_3' instead of creating a new one.


Training until validation scores don't improve for 454 rounds
[50]	valid_0's total_error: 163.88


[I 2025-07-13 12:36:39,196] Trial 9 finished with value: 1.5077238552994643 and parameters: {'tweedie_variance_power': 1.8680138426687347, 'learning_rate': 0.07295608449546184, 'num_leaves': 256, 'lambda_l1': 2.2006730056278445, 'lambda_l2': 3.6105635460300163, 'min_data_in_leaf': 74, 'feature_fraction': 0.9968101525801872, 'bagging_fraction': 0.6581734888953041, 'bagging_freq': 2}. Best is trial 6 with value: 0.2708316678474519.


[100]	valid_0's total_error: 20956.4
Did not meet early stopping. Best iteration is:
[1]	valid_0's total_error: 1.50808
Evaluated only: total_error
Training until validation scores don't improve for 502 rounds
[50]	valid_0's total_error: 0.389893


[W 2025-07-13 12:36:46,909] Trial 10 failed with parameters: {'tweedie_variance_power': 1.40718400812128, 'learning_rate': 0.038846735520867384, 'num_leaves': 103, 'lambda_l1': 7.096515628784759, 'lambda_l2': 9.00142430623231, 'min_data_in_leaf': 54, 'feature_fraction': 0.32256438841895063, 'bagging_fraction': 0.8359032812885376, 'bagging_freq': 6} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_301963/3893123424.py", line 62, in objective
    model = lgb.train(
            ^^^^^^^^^^
  File "/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/lightgbm/engine.py", line 322, in train
    booster.update(fobj=fobj)
  File "/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/lightgbm/basic.py", line 4155, in upda

KeyboardInterrupt: 

In [ ]:
best_params = study.best_params
best_params.update({
    'objective': 'tweedie',
    'boosting_type': 'gbdt',
    'force_row_wise': True,
    'metric': 'None',
    'extra_trees': True,
    'first_metric_only': True,
    "max_depth": -1,
    "device": "cuda",
    "feature_pre_filter": False,
})


avg_best_iter = int(study.best_trial.user_attrs["avg_num_iterations"])
print(f"🔍 Mejor total_error: {study.best_value:.5f}")
print(f"🏁 Mejor iteración promedio: {avg_best_iter}")
print(f"📊 Mejor conjunto de parámetros: {best_params}")

# Entrenar modelo final con todo el dataset
final_df = df.groupby(["product_id", "customer_id"]).filter(lambda x: len(x) >= 12)
final_train = lgb.Dataset(
    final_df.drop(columns=[col for col in drop_cols if col in final_df.columns]),
    label=final_df["target"],
    categorical_feature="auto",
    #weight=(final_df["tn"] + 0.1),
)

final_model = lgb.train(
    params=best_params,
    train_set=final_train,
    num_boost_round=avg_best_iter,
    callbacks=[lgb.log_evaluation(period=10)],
)

🔍 Mejor total_error: 0.20581
🏁 Mejor iteración promedio: 1309
📊 Mejor conjunto de parámetros: {'tweedie_variance_power': 1.4043923777832485, 'learning_rate': 0.04993830362666026, 'num_leaves': 209, 'lambda_l1': 8.462907381395699, 'lambda_l2': 8.10925858207787, 'min_data_in_leaf': 54, 'feature_fraction': 0.631776952562918, 'bagging_fraction': 0.885205173814044, 'bagging_freq': 1, 'objective': 'tweedie', 'boosting_type': 'gbdt', 'force_row_wise': True, 'metric': 'None', 'extra_trees': True, 'first_metric_only': True, 'max_depth': -1, 'device': 'cuda', 'feature_pre_filter': False}


In [ ]:
features = final_train.feature_name
# replace spaces in final_test_df columns names by _ (underscore)
final_test_df.columns = final_test_df.columns.str.replace(" ", "_", regex=False)

# Reemplazo de inf por nan
final_test_df.replace([np.inf, -np.inf], np.nan, inplace=True)



# Transformar object a category (menos claves)
columns = final_test_df.select_dtypes(include=["object"]).columns.tolist()
for col in columns:
    if col not in ["product_id", "customer_id", "date_id"]:
        final_test_df[col] = final_test_df[col].astype("category")

# Eliminar columnas datetime innecesarias
datetime_cols = final_test_df.select_dtypes(include=["datetime"]).columns.tolist()
for col in datetime_cols:
    if col != "date_id":
        final_test_df.drop(columns=[col], inplace=True)



final_predictions = final_model.predict(final_test_df[features])
final_test_df["predictions"] = final_predictions
# agrupo por product_id y customer_id
final_test_df_grouped = final_test_df.groupby("product_id").agg({
    "predictions": "sum",
}).reset_index()
final_test_df_grouped = final_test_df_grouped[final_test_df_grouped["product_id"].isin(product_ids)]
submission = final_test_df_grouped[["product_id", "predictions"]].copy()
submission.rename(columns={"predictions": "tn"}, inplace=True)
submission["tn"] = submission["tn"].clip(lower=0)  # Asegurar que no haya valores negativos
submission.to_csv("submission_lgb.csv", index=False)
submission

,product_id,tn
0,20001,1446.246083
1,20002,1026.723953
2,20003,804.005776
3,20004,545.402825
4,20005,538.188319
...,...,...
920,21263,0.016726
922,21265,0.018908
923,21266,0.021311
924,21267,0.017471
